# imports

In [ ]:
import time
import json
import pathlib
import warnings
import random
import os
import math

import numpy as np
import pandas as pd
import torch
import optuna

from sklearn.metrics import root_mean_squared_error
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MSE
from neuralforecast.models import PatchTST

torch.set_float32_matmul_precision("medium")

# ============================================================
# HELPERS
# ============================================================
def build_single_series_nf_df(df_all, home_col):
    """
    Convert one household column into NeuralForecast format:
    unique_id, ds, y
    """
    if not isinstance(df_all.index, pd.DatetimeIndex):
        raise ValueError("df_all index must be a DatetimeIndex.")

    df_nf = pd.DataFrame({
        "unique_id": home_col,
        "ds": df_all.index,
        "y": df_all[home_col].values
    }).reset_index(drop=True)

    return df_nf


def split_train_val_test_single(df_nf, test_start, forecast_horizon, training_size, val_days=3, freq_minutes=15):
    """
    Split a single-series NeuralForecast dataframe into train / validation / test.
    """
    test_start = pd.Timestamp(test_start)
    val_start = test_start - pd.Timedelta(days=val_days)
    test_end = test_start + pd.Timedelta(minutes=freq_minutes * forecast_horizon)

    expected_val_len = val_days * (24 * 60 // freq_minutes)

    df_nf = df_nf.sort_values("ds").reset_index(drop=True)

    train_candidates = df_nf[df_nf["ds"] < val_start].copy()
    train_df = train_candidates.iloc[-training_size:].copy()

    val_df = df_nf[(df_nf["ds"] >= val_start) & (df_nf["ds"] < test_start)].copy()
    test_df = df_nf[(df_nf["ds"] >= test_start) & (df_nf["ds"] < test_end)].copy()

    if len(train_df) != training_size:
        raise ValueError(f"Expected {training_size} training rows, got {len(train_df)}")

    if len(val_df) != expected_val_len:
        raise ValueError(f"Expected {expected_val_len} validation rows, got {len(val_df)}")

    if len(test_df) != forecast_horizon:
        raise ValueError(f"Expected {forecast_horizon} test rows, got {len(test_df)}")

    return train_df, val_df, test_df


def rolling_forecasting_validation_predictions_single(train_df, val_df, h, model_params, freq="15min"):
    """
    Rolling validation for one series only.
    """
    rolling_train_df = train_df.copy()
    val_predictions = []

    val_starts = sorted(val_df["ds"].unique())[::h]

    for window_start in val_starts:
        model = PatchTST(
            h=h,
            input_size=model_params["input_size"],
            patch_len=model_params["patch_len"],
            stride=model_params["stride"],
            hidden_size=model_params["hidden_size"],
            n_heads=model_params["n_heads"],
            batch_size=model_params["batch_size"],
            learning_rate=model_params["learning_rate"],
            max_steps=MAX_STEPS,
            val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
            random_seed=42,
            loss=MSE(),
        )

        nf = NeuralForecast(models=[model], freq=freq)
        nf.fit(df=rolling_train_df)

        preds = nf.predict()
        val_predictions.append(preds)

        next_val_chunk = val_df[
            (val_df["ds"] >= window_start) &
            (val_df["ds"] < window_start + pd.Timedelta(minutes=15 * h))
        ].copy()

        rolling_train_df = pd.concat([rolling_train_df, next_val_chunk], ignore_index=True)
        rolling_train_df = rolling_train_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    return pd.concat(val_predictions, ignore_index=True)


def compute_rmse_single(val_df, val_preds_df, pred_col="PatchTST"):
    """
    Compute RMSE for one household.
    """
    val_compare_df = val_df.merge(val_preds_df, on=["unique_id", "ds"], how="left")
    rmse = root_mean_squared_error(val_compare_df["y"], val_compare_df[pred_col])
    return rmse, val_compare_df


def make_objective_single(train_df, val_df, forecast_horizon):
    """
    Create Optuna objective for one household.
    """
    def objective(trial):
        model_params = {
            "input_size": trial.suggest_categorical("input_size", [
                forecast_horizon * 2,
                forecast_horizon * 3,
                forecast_horizon * 4,
            ]),
            "patch_len": trial.suggest_categorical("patch_len", [8, 12, 16, 24]),
            "stride": trial.suggest_categorical("stride", [4, 8, 12]),
            "hidden_size": trial.suggest_categorical("hidden_size", [16, 32, 64, 128]),
            "n_heads": trial.suggest_categorical("n_heads", [2, 4, 8]),
            "batch_size": trial.suggest_categorical("batch_size", [16, 32, 64]),
            "learning_rate": trial.suggest_float("learning_rate", 5e-4, 2e-3, log=True),
        }

        try:
            val_preds_df = rolling_forecasting_validation_predictions_single(
                train_df=train_df,
                val_df=val_df,
                h=forecast_horizon,
                model_params=model_params,
                freq="15min"
            )

            rmse, _ = compute_rmse_single(
                val_df=val_df,
                val_preds_df=val_preds_df,
                pred_col="PatchTST"
            )

            return rmse

        except Exception as e:
            print(f"Trial failed: {e}")
            return float("inf")

    return objective



# All countries

In [ ]:

import time

start_time = time.time()

project_path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone"
days_json_path = pathlib.Path(project_path) / "dataset_days.json"

#countries = ["Germany", "Ireland", "Portugal"]
#countries = ["Germany"]
countries = ["Germany", "Ireland", "Portugal","Denmark"]

#days = ["day1", "day2", "day3", "day4", "day5"]
days = ["day1"]



weather_cols = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
#    "price_eur_kwh"
]



# ============================================================
# GLOBAL SETTINGS
# ============================================================
MAX_STEPS = 500
VAL_CHECK_STEPS = MAX_STEPS // 10

forecast_horizon = 96
training_size = 96 * 7 * 3 * 2
opt_trials = 10








# ============================================================
# MAIN
# ============================================================
start_time = time.time()

with open(days_json_path, "r") as f:
    dataset_days = json.load(f)

# -------------------------
# Loop over countries
# -------------------------
for country in countries:
    country_start_time = time.time()

    print(f"\n{'#'*100}")
    print(f"COUNTRY: {country}")
    print(f"{'#'*100}")

    if country == "Denmark" and "price_eur_kwh" not in weather_cols:

        weather_cols.append("price_eur_kwh")

    dataset_path = pathlib.Path(project_path) / "DataCleaning" / "clean" / f"dataset_{country}.csv"

    df_all = pd.read_csv(dataset_path, parse_dates=["timestamp"])
    df_all = df_all.set_index("timestamp")
    df_all = df_all.sort_index()

    home_cols = [col for col in df_all.columns if col.startswith("home_")]

    print(f"Detected {len(home_cols)} homes for {country}.")
    print(home_cols)

    for day_name in days:
        selected_day = dataset_days[country][day_name]
        date = f"{selected_day} 00:00:00"
        forecast_end_date = str(pd.Timestamp(date) + pd.Timedelta(days=1))

        print(f"\n{'='*100}")
        print(f"Running {country} - {day_name}")
        print(f"Forecast start: {date}")
        print(f"Forecast end:   {forecast_end_date}")
        print(f"{'='*100}")

        all_home_preds = []

        for home_col in home_cols:
            print(f"\n--- Training local PatchTST for {home_col} ---")

            df_home_nf = build_single_series_nf_df(df_all, home_col)

            train_df, val_df, test_df = split_train_val_test_single(
                df_nf=df_home_nf,
                test_start=date,
                forecast_horizon=forecast_horizon,
                training_size=training_size,
                val_days=3,
                freq_minutes=15
            )

            for name, df_part in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
                start = df_part["ds"].min()
                end = df_part["ds"].max()
                print(f"{name}: {start} to {end} (Shape: {df_part.shape})")

            objective = make_objective_single(
                train_df=train_df,
                val_df=val_df,
                forecast_horizon=forecast_horizon
            )

            study = optuna.create_study(direction="minimize")
            study.optimize(objective, n_trials=opt_trials, show_progress_bar=True)

            print(f"Best RMSE for {home_col}: {study.best_value}")
            print(f"Best params for {home_col}: {study.best_params}")

            best_params = study.best_params

            train_val_df = pd.concat([train_df, val_df], ignore_index=True)
            train_val_df = train_val_df.sort_values(["unique_id", "ds"]).reset_index(drop=True)

            final_patchtst = PatchTST(
                h=forecast_horizon,
                input_size=best_params["input_size"],
                patch_len=best_params["patch_len"],
                stride=best_params["stride"],
                hidden_size=best_params["hidden_size"],
                n_heads=best_params["n_heads"],
                batch_size=best_params["batch_size"],
                learning_rate=best_params["learning_rate"],
                max_steps=MAX_STEPS,
                val_check_steps=min(VAL_CHECK_STEPS, MAX_STEPS),
                random_seed=42,
                loss=MSE()
            )

            nf_final = NeuralForecast(
                models=[final_patchtst],
                freq="15min"
            )

            nf_final.fit(df=train_val_df)

            test_preds_df = nf_final.predict()

            # keep only ds and prediction, rename to household column name
            home_preds = test_preds_df[["ds", "PatchTST"]].copy()
            home_preds = home_preds.rename(columns={"PatchTST": home_col})
            home_preds = home_preds.set_index("ds")

            all_home_preds.append(home_preds)

        final_test_preds_wide = pd.concat(all_home_preds, axis=1).sort_index()




        # ========================================================
        # COUNTRY RUNTIME
        # ========================================================
        country_runtime = time.time() - country_start_time

        print(
            f"\nTotal runtime for {country}: "
            f"{country_runtime:.2f} seconds"
        )

        # ========================================================
        # SAVE / UPDATE JSON WITHOUT OVERWRITING EXISTING CONTENT
        # ========================================================
        json_path = pathlib.Path(project_path) / "Outputs" / f"time_spend_{country}.json"

        if json_path.exists():
            with open(json_path, "r") as f:
                runtime_dict = json.load(f)
        else:
            runtime_dict = {}

        if "Local" not in runtime_dict:
            runtime_dict["Local"] = {}

        runtime_dict["Local"]["PatchTST"] = country_runtime

        with open(json_path, "w") as f:
            json.dump(runtime_dict, f, indent=4)

        print(f"Saved/updated runtime JSON: {json_path}")






        save_dir = pathlib.Path(project_path) / "Outputs" / "Local models" / "PatchTST"
        save_dir.mkdir(parents=True, exist_ok=True)

        save_path = save_dir / f"prediction_PatchTST_local_{day_name}_{country}.csv"
        final_test_preds_wide.to_csv(save_path, index=True)

        print(f"Saved final_test_preds_wide to: {save_path}")

end_time = time.time()
total_seconds = end_time - start_time

print(f"\nTotal runtime: {total_seconds:.2f} seconds")

# end 